# Test: Training Data Insertions for OLMo-Core

Two tests:
1. Raw token insertion — verify tokens are inserted at the correct training-order position
2. Text string insertion — encode/decode round-trip with the dolma2 tokenizer

In [ ]:
import os
import tempfile
import numpy as np
import torch

from pretrain_experiments.insertion_map import InsertionMapWriter, InsertionMapReader
from olmo_core.data import NumpyFSLDatasetConfig, NumpyDataLoaderConfig, TokenizerConfig

## Test 1: Raw Token Insertion

In [ ]:
tmpdir = tempfile.mkdtemp()
print(f"Working directory: {tmpdir}")

# Create a small .npy dataset: 1000 sequences x 512 tokens, all ones
num_sequences = 1000
seq_length = 512
data = np.ones(num_sequences * seq_length, dtype=np.uint32)
data_path = os.path.join(tmpdir, "test_data.npy")
np.save(data_path, data)
print(f"Created test data: {data.shape} tokens at {data_path}")

In [ ]:
# Create an insertion map: insert tokens [99, 98, 97] at position 10
# in training-order index 5 (the 6th sequence seen during training)
INSERTION_TRAINING_IDX = 5
INSERTION_POS = 10
INSERTION_TOKENS = [99, 98, 97]

insertion_map_path = os.path.join(tmpdir, "insertion_map.h5")
optimized_path = os.path.join(tmpdir, "insertion_map_optimized.h5")

writer = InsertionMapWriter(insertion_map_path)
writer.write_dict({
    INSERTION_TRAINING_IDX: [(INSERTION_POS, INSERTION_TOKENS)]
})
writer.save_optimized(optimized_path)
print(f"Created insertion map at {optimized_path}")

In [ ]:
# Helper to build dataset and data loader
def build_data_loader(data_path, tmpdir, seq_length):
    tokenizer_config = TokenizerConfig.dolma2()
    dataset_config = NumpyFSLDatasetConfig(
        tokenizer=tokenizer_config,
        paths=[data_path],
        work_dir=os.path.join(tmpdir, "work_dir"),
        sequence_length=seq_length,
    )
    dataset = dataset_config.build()

    data_loader_config = NumpyDataLoaderConfig(
        global_batch_size=seq_length * 512,
        seed=42,
    )
    return data_loader_config.build(dataset)

In [ ]:
# Test WITHOUT insertions — record original data at training-order position 5
os.environ.pop("OLMO_CORE_INSERTION_MAP_FILE", None)

loader_no_insert = build_data_loader(data_path, tmpdir, seq_length)
loader_no_insert.reshuffle(epoch=1)

# Get the global indices to find which dataset index is at training position 5
global_indices = loader_no_insert.get_global_indices()
dataset_idx_for_pos5 = int(global_indices[INSERTION_TRAINING_IDX])
print(f"Training-order index {INSERTION_TRAINING_IDX} maps to dataset index {dataset_idx_for_pos5}")

# Read the original item
original_item = loader_no_insert._get_dataset_item(dataset_idx_for_pos5)
original_tokens = original_item["input_ids"][INSERTION_POS:INSERTION_POS+len(INSERTION_TOKENS)].tolist()
print(f"Original tokens at position {INSERTION_POS}: {original_tokens}")
assert original_tokens == [1, 1, 1], f"Expected all 1s, got {original_tokens}"

In [ ]:
# Test WITH insertions
os.environ["OLMO_CORE_INSERTION_MAP_FILE"] = optimized_path

loader_with_insert = build_data_loader(data_path, tmpdir, seq_length)
loader_with_insert.reshuffle(epoch=1)

# Verify the remapping happened
print(f"Dataset insertions: {loader_with_insert._dataset_insertions}")
assert dataset_idx_for_pos5 in loader_with_insert._dataset_insertions, \
    f"Expected dataset index {dataset_idx_for_pos5} in insertions"

# Read the modified item
modified_item = loader_with_insert._get_dataset_item(dataset_idx_for_pos5)
modified_tokens = modified_item["input_ids"][INSERTION_POS:INSERTION_POS+len(INSERTION_TOKENS)].tolist()
print(f"Modified tokens at position {INSERTION_POS}: {modified_tokens}")
assert modified_tokens == INSERTION_TOKENS, f"Expected {INSERTION_TOKENS}, got {modified_tokens}"

# Verify OTHER positions are unchanged
assert modified_item["input_ids"][0].item() == 1, "Token at position 0 should be unchanged"
assert modified_item["input_ids"][INSERTION_POS + len(INSERTION_TOKENS)].item() == 1, \
    "Token after insertion should be unchanged"

print("\n=== Test 1 PASSED: Raw token insertion works correctly ===")

# Clean up env var
os.environ.pop("OLMO_CORE_INSERTION_MAP_FILE", None)

## Test 2: Text String Insertion (Encode/Decode Round-Trip)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("allenai/dolma2-tokenizer")

text = "The quick brown fox jumps over the lazy dog."
token_ids = tokenizer.encode(text)
print(f"Text: {text!r}")
print(f"Token IDs ({len(token_ids)} tokens): {token_ids}")
print(f"Decoded back: {tokenizer.decode(token_ids)!r}")

In [ ]:
# Create insertion map with text tokens at training-order index 10, position 0
TEXT_TRAINING_IDX = 10
TEXT_POS = 0

tmpdir2 = tempfile.mkdtemp()
text_insertion_path = os.path.join(tmpdir2, "text_insertion.h5")
text_optimized_path = os.path.join(tmpdir2, "text_insertion_optimized.h5")

writer2 = InsertionMapWriter(text_insertion_path)
writer2.write_dict({
    TEXT_TRAINING_IDX: [(TEXT_POS, token_ids)]
})
writer2.save_optimized(text_optimized_path)

# Create dataset (reuse data from test 1)
data_path2 = os.path.join(tmpdir2, "test_data.npy")
np.save(data_path2, data)

In [ ]:
# Build loader with text insertions
os.environ["OLMO_CORE_INSERTION_MAP_FILE"] = text_optimized_path

loader_text = build_data_loader(data_path2, tmpdir2, seq_length)
loader_text.reshuffle(epoch=1)

# Find which dataset index corresponds to training-order index 10
global_indices2 = loader_text.get_global_indices()
dataset_idx_for_text = int(global_indices2[TEXT_TRAINING_IDX])
print(f"Training-order index {TEXT_TRAINING_IDX} maps to dataset index {dataset_idx_for_text}")

# Get the modified item
text_item = loader_text._get_dataset_item(dataset_idx_for_text)
inserted_tokens = text_item["input_ids"][TEXT_POS:TEXT_POS+len(token_ids)].tolist()
decoded = tokenizer.decode(inserted_tokens)

print(f"Inserted tokens: {inserted_tokens}")
print(f"Decoded text: {decoded!r}")
print(f"Original text: {text!r}")

assert decoded == text, f"Decoded text {decoded!r} does not match original {text!r}"
assert inserted_tokens == token_ids, f"Token mismatch: {inserted_tokens} vs {token_ids}"

print("\n=== Test 2 PASSED: Text encode/decode round-trip works correctly ===")

# Clean up
os.environ.pop("OLMO_CORE_INSERTION_MAP_FILE", None)

In [ ]:
# Cleanup temp directories
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
shutil.rmtree(tmpdir2, ignore_errors=True)
print("Cleaned up temp directories.")
print("\n=== All tests passed! ===")